# 13.3 模型給整段回答多大機率？

# 第 13 章：把回答偏好變成訓練

前置：第7章SFT，以及第8–9章的「好回答」規則。可以直接走這條路，不必先學圖片。把兩篇答案像兩道菜一起試喫：喜歡哪一篇，與「兩篇是否都安全」是不同問題。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：試喫兩道菜：比起原廚師，更偏向好回答**

chosen 與 rejected 必須對同一個問題。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/dpo.svg")))

**先想一想：**更長答案分數較負，代表必然比較差嗎？

整篇答案的機率是每一步在已有前文下的條件機率相乘；取log後變成相加，比直接乘小數穩定。

**把直覺寫成數學：**log p(answer|prompt)=Σ有效答案格 log p(token_t|prefix_t)；採sum，不能悄悄改成mean。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.alignment import sequence_log_probability

logits = torch.tensor([[[0.0, 2.0, 0.0], [2.0, 0.0, 0.0], [0.0, 0.0, 2.0]]])
labels = torch.tensor([[-100, 0, 2]])
score = sequence_log_probability(logits, labels)
print(score)
assert torch.allclose(score, logits.log_softmax(-1)[0, 1, 0] + logits.log_softmax(-1)[0, 2, 2])

**如何讀結果：**總分自然含長度；偏好配對與reference差分纔是後面要比較的量。

**只改一件事：**只增加一個有效token，觀察總分與平均分區別。
